We have forecasted for product 001, the next 14 values of the unit's sold, now we will move on to generate a reorder data basing on that forecast.Remember we used the Holt winter model. So unlike the moving average, values are not constant across all the 14 days.

In [2]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv(r"C:\Users\ahmed\Documents\Data Science LUXDEV\PPORTFOLIO PROJECTS\INVENTORY FORECASTING\CSV\CLEAN\forecasted_units_sold_SKU001.csv")
df.head()

,Date,units_sold
0,2026-01-01,46.0
1,2026-01-02,45.0
2,2026-01-03,55.0
3,2026-01-04,53.0
4,2026-01-05,45.0


The `units_sold` are forecasted values and vary on each day due to our holt winter method. To generate a reorder logic we can use the product master  data for specifically `SKU001`.The data we want is the `lead_time_days` and `safety_stock_days`.

In [4]:
df_product_master = pd.read_csv(r"C:\Users\ahmed\Documents\Data Science LUXDEV\PPORTFOLIO PROJECTS\INVENTORY FORECASTING\CSV\CLEAN\products_cleaned.csv")
df_product_master

,product_id,product_name,category,unit_cost,unit_price,lead_time_days,safety_stock_days
0,SKU001,Soda 500ml,Beverages,126.94,154.61,5,4
1,SKU002,Bottled Water 1L,Beverages,119.13,96.01,6,4
2,SKU003,Juice 1L,Beverages,125.64,213.63,6,2
3,SKU004,Energy Drink 250ml,Beverages,93.94,143.04,3,4
4,SKU005,Iced Tea 500ml,Beverages,113.67,219.87,6,3
5,SKU006,Potato Crisps,Snacks,50.45,133.19,9,2
6,SKU007,Chocolate Bar,Snacks,104.49,144.75,3,4
7,SKU008,Biscuits Pack,Snacks,61.91,195.60,5,4
8,SKU009,Peanuts 200g,Snacks,100.05,79.20,4,3
9,SKU010,Popcorn Bag,Snacks,33.94,73.14,7,4


for SKU001 , lead_time_days = 5, safety_stock_days = 4. We will enrich our forecast data with these two columns, NOTE the values across all the 14 days will be constant for lead_time_days and safety_stock_days. Our re_order point logic will be whenever the current stock is less than (`demand_during_leadtime` + `safety_stock`) .

**columns**
1. `lead_time_days`
2. `safety_stock_days`
3. `demand_during_leadtime` = `units_sold` * `lead_time_days`
4. `safety_stock` = `units_sold` * `safety_stock_days`
5. `ROP` = (`demand_during_leadtime` + `safety_stock`)
6. `current_stock`

**NOTE:** We also want data for the current stock of that specific SKU.We will find that in our sales data

In [5]:
df_sales=pd.read_csv(r"C:\Users\ahmed\Documents\Data Science LUXDEV\PPORTFOLIO PROJECTS\INVENTORY FORECASTING\CSV\CLEAN\feature_engineered_sales.csv")
df_sales

,product_id,product_name_x,category_x,units_sold,unit_price_x,promotion_flag,stock_level_end_of_day,stockout_flag,date,product_name_y,...,Month,Quarter,is_festive,day_of_year,Revenue,rolling_7d_avg,rolling_28d_avg,previous_day_sale,previous_week_sale,days_since_last_promo
0,SKU001,Soda 500ml,Beverages,29,154.61,False,365.9,False,2024-01-01,Soda 500ml,...,1,1,False,1,4483.69,29.000000,29.000000,NaN,NaN,0
1,SKU001,Soda 500ml,Beverages,27,154.61,False,338.9,False,2024-01-02,Soda 500ml,...,1,1,False,2,4174.47,28.000000,28.000000,29.0,NaN,0
2,SKU001,Soda 500ml,Beverages,25,154.61,False,313.9,False,2024-01-03,Soda 500ml,...,1,1,False,3,3865.25,27.000000,27.000000,27.0,NaN,0
3,SKU001,Soda 500ml,Beverages,33,154.61,True,280.9,False,2024-01-04,Soda 500ml,...,1,1,False,4,5102.13,28.500000,28.500000,25.0,NaN,0
4,SKU001,Soda 500ml,Beverages,26,154.61,False,254.9,False,2024-01-05,Soda 500ml,...,1,1,False,5,4019.86,28.000000,28.000000,33.0,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18270,SKU025,Sukuma Wiki Bunch,Produce,75,43.39,False,716.8,False,2025-12-27,Sukuma Wiki Bunch,...,12,4,True,361,3254.25,71.571429,72.035714,85.0,88.0,0
18271,SKU025,Sukuma Wiki Bunch,Produce,87,43.39,False,629.8,False,2025-12-28,Sukuma Wiki Bunch,...,12,4,True,362,3774.93,71.857143,71.500000,75.0,85.0,0
18272,SKU025,Sukuma Wiki Bunch,Produce,58,43.39,False,571.8,False,2025-12-29,Sukuma Wiki Bunch,...,12,4,True,363,2516.62,70.285714,70.678571,87.0,69.0,0
18273,SKU025,Sukuma Wiki Bunch,Produce,79,43.39,False,492.8,False,2025-12-30,Sukuma Wiki Bunch,...,12,4,True,364,3427.81,72.000000,70.392857,58.0,67.0,0


In [6]:
df_soda = df_sales[df_sales['product_id']=='SKU001'].copy()

In [7]:
df_soda

,product_id,product_name_x,category_x,units_sold,unit_price_x,promotion_flag,stock_level_end_of_day,stockout_flag,date,product_name_y,...,Month,Quarter,is_festive,day_of_year,Revenue,rolling_7d_avg,rolling_28d_avg,previous_day_sale,previous_week_sale,days_since_last_promo
0,SKU001,Soda 500ml,Beverages,29,154.61,False,365.9,False,2024-01-01,Soda 500ml,...,1,1,False,1,4483.69,29.000000,29.000000,NaN,NaN,0
1,SKU001,Soda 500ml,Beverages,27,154.61,False,338.9,False,2024-01-02,Soda 500ml,...,1,1,False,2,4174.47,28.000000,28.000000,29.0,NaN,0
2,SKU001,Soda 500ml,Beverages,25,154.61,False,313.9,False,2024-01-03,Soda 500ml,...,1,1,False,3,3865.25,27.000000,27.000000,27.0,NaN,0
3,SKU001,Soda 500ml,Beverages,33,154.61,True,280.9,False,2024-01-04,Soda 500ml,...,1,1,False,4,5102.13,28.500000,28.500000,25.0,NaN,0
4,SKU001,Soda 500ml,Beverages,26,154.61,False,254.9,False,2024-01-05,Soda 500ml,...,1,1,False,5,4019.86,28.000000,28.000000,33.0,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
726,SKU001,Soda 500ml,Beverages,71,154.61,False,60.8,False,2025-12-27,Soda 500ml,...,12,4,True,361,10977.31,48.714286,46.750000,35.0,67.0,0
727,SKU001,Soda 500ml,Beverages,66,154.61,False,651.2,False,2025-12-28,Soda 500ml,...,12,4,True,362,10204.26,50.000000,47.285714,71.0,57.0,0
728,SKU001,Soda 500ml,Beverages,46,154.61,False,605.2,False,2025-12-29,Soda 500ml,...,12,4,True,363,7112.06,49.285714,47.607143,66.0,51.0,0
729,SKU001,Soda 500ml,Beverages,45,154.61,False,560.2,False,2025-12-30,Soda 500ml,...,12,4,True,364,6957.45,48.714286,47.535714,46.0,49.0,0


So the `current stock` is basically the last value in the `stock_level_end_of_day` column in our sales data

In [8]:

df_soda.iloc[-1:]

,product_id,product_name_x,category_x,units_sold,unit_price_x,promotion_flag,stock_level_end_of_day,stockout_flag,date,product_name_y,...,Month,Quarter,is_festive,day_of_year,Revenue,rolling_7d_avg,rolling_28d_avg,previous_day_sale,previous_week_sale,days_since_last_promo
730,SKU001,Soda 500ml,Beverages,41,154.61,False,519.2,False,2025-12-31,Soda 500ml,...,12,4,True,365,6339.01,49.142857,47.357143,45.0,38.0,0


## enriching our data

In [9]:
df['lead_time_days'] = 5
df['safety_stock_days'] = 4

In [10]:
df['safety_stock'] = df['safety_stock_days']*df['units_sold']
df['demand_during_leadtime'] = df['lead_time_days']*df['units_sold']
df['current_stock'] = 519.2
df['ROP'] = df['demand_during_leadtime']+df['safety_stock']

In [11]:
df

,Date,units_sold,lead_time_days,safety_stock_days,safety_stock,demand_during_leadtime,current_stock,ROP
0,2026-01-01,46.0,5,4,184.0,230.0,519.2,414.0
1,2026-01-02,45.0,5,4,180.0,225.0,519.2,405.0
2,2026-01-03,55.0,5,4,220.0,275.0,519.2,495.0
3,2026-01-04,53.0,5,4,212.0,265.0,519.2,477.0
4,2026-01-05,45.0,5,4,180.0,225.0,519.2,405.0
5,2026-01-06,47.0,5,4,188.0,235.0,519.2,423.0
6,2026-01-07,47.0,5,4,188.0,235.0,519.2,423.0
7,2026-01-08,46.0,5,4,184.0,230.0,519.2,414.0
8,2026-01-09,46.0,5,4,184.0,230.0,519.2,414.0
9,2026-01-10,55.0,5,4,220.0,275.0,519.2,495.0


We now create a reorder_flag where if current stock is less or equal to ROP, we flag that SKU for reorder.So far from our data we have no ROP exceeding the current stock so the flag should be `False`, irregardless we will create a code that generates the flag as per our reorder logic.

In [12]:
target_cover_days = 14
current_stock = df['current_stock']
daily_forecast = df['units_sold']
lead_time_days = df['lead_time_days']
demand_during_leadtime = df['demand_during_leadtime']

days_of_cover = current_stock/ daily_forecast

def decision(rop):
    reorder_now = current_stock <= rop
    target_stock = ((lead_time_days +target_cover_days)*daily_forecast) + (rop-demand_during_leadtime) #covers stock for the next 14 days + safety_stock_days+ lead_time_days
    
    qty = np.where(
        reorder_now,
        np.maximum(0,np.ceil(target_stock-current_stock)),0
    )
    

    return reorder_now, qty

ROP = df['ROP']
x = decision(ROP)
print(x)

(0     False
1     False
2     False
3     False
4     False
5     False
6     False
7     False
8     False
9     False
10    False
11    False
12    False
13    False
dtype: bool, array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]))


In [13]:
df

,Date,units_sold,lead_time_days,safety_stock_days,safety_stock,demand_during_leadtime,current_stock,ROP
0,2026-01-01,46.0,5,4,184.0,230.0,519.2,414.0
1,2026-01-02,45.0,5,4,180.0,225.0,519.2,405.0
2,2026-01-03,55.0,5,4,220.0,275.0,519.2,495.0
3,2026-01-04,53.0,5,4,212.0,265.0,519.2,477.0
4,2026-01-05,45.0,5,4,180.0,225.0,519.2,405.0
5,2026-01-06,47.0,5,4,188.0,235.0,519.2,423.0
6,2026-01-07,47.0,5,4,188.0,235.0,519.2,423.0
7,2026-01-08,46.0,5,4,184.0,230.0,519.2,414.0
8,2026-01-09,46.0,5,4,184.0,230.0,519.2,414.0
9,2026-01-10,55.0,5,4,220.0,275.0,519.2,495.0


Now this table gives a demand forecast for the next 14 days.We can speculating how many units we need to have to meet the demand of this product for the next 14 days also considering the lead time days and safety stock day.

In [14]:
daily_average=df['units_sold'].mean()
print(round(daily_average,0))

48.0


In [20]:
demand_inventory=daily_average * (target_cover_days + 9)
print(round(demand_inventory,0))

1114.0


So for the next 14 days, we need to have atleast `1114` units to safely last us that long considering the fact that on a daily basis an average of 48 units are sold , and we have a leadtime of 5 days and an extra 4 days for contingency. 

In [18]:
average_rop = df['ROP'].mean()
average_rop

np.float64(435.85714285714283)

Our **ROP** stands at `436`, meaning whenever the current stock is below or equal to that value, we should reorder immediately to keep up with the demand.